In [ ]:
import shutil
import random
import yaml
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))
data_dir = Path("../data")
import json

In [ ]:
def yolo_structure(data_dir, output_dir, split_ratio=(0.7, 0.2, 0.1)):
    flight_dir = [d for d in data_dir.iterdir() if d.is_dir()]
    flight_ids = [d.name for d in flight_dir]
    total_flights = len(flight_ids)
    train_end = int(total_flights * split_ratio[0])
    val_end = train_end + int(total_flights * split_ratio[1])
    splits = {'train': flight_ids[:train_end], 'val': flight_ids[train_end:val_end], 'test': flight_ids[val_end:]}

    for split, ids in splits.items():
        (output_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / split / 'labels').mkdir(parents=True, exist_ok=True)
        for flight_id in ids:
            flight_labels_dir = data_dir / "labels" / flight_id
            flight_images_dir = data_dir / "frames_detection" / flight_id / "thermal"

            for label_file in flight_labels_dir.glob("*.txt"):
                image_file = flight_images_dir / f"{label_file.stem}.png"

                if image_file.exists():
                    shutil.copy(label_file, output_dir / split / "labels" / label_file.name)
                    shutil.copy(image_file, output_dir / split / "images" / image_file.name)

    yaml_content = {
        "path": str(output_dir.absolute()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {
            0: "Wild boar",
            1: "Red deer",
            2: "Roe deer"
        }
    }
    yaml_path = output_dir / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_content, f, sort_keys=False)

In [ ]:
yolo_structure(data_dir, data_dir / 'yolo_data')

In [ ]:
# 1. Count Total Labels for Flight 0
labels_0 = list((data_dir / "labels" / "0").glob("*.txt"))
print(f"Total Label files for Flight 0: {len(labels_0)}")

images_0 = list((data_dir / "frames_detection" / "0" / "thermal").glob("*.png"))
print(f"Total Image files for Flight 0: {len(images_0)}")

# 3. Check for Overlap (The "Match" Test)
matches = 0
for label in labels_0:
    expected_image = data_dir / "frames_detection" / "0" / "thermal" / f"{label.stem}.png"
    if expected_image.exists():
        matches += 1

print(f"\nTotal Overlapping Matches: {matches}")

In [ ]:
def coco_split(json_path, yolo_path, output_path):
    with open(json_path) as f:
        coco_data = json.load(f)

    splits = {'train': [], 'val': [], 'test': []}
    for split in splits:
        yolo_split_dir = yolo_path / split / 'images'
        if yolo_split_dir.exists():
            splits[split] = [img.name for img in yolo_split_dir.glob("*.png")]    